# Pascal VOC 2007 Semantic Segmentation (v4)

**Changes from v2:**
- Fixed mDice metric (global accumulation instead of per-image averaging)
- DeepLabV3+ output stride 8 (was 16) for finer spatial detail
- SAM2 decoder: 3-level FPN fusion + stronger head + dropout
- CE+Dice loss (0.7+0.3) for all models — more stable than Dice-only
- Image size 512 for UNet/DeepLabV3+/SAM2 (DINOv2 stays 224)
- SAM2 eval now passes `--sam2-cfg` to avoid config mismatch

## 0. Setup & Dataset

In [ ]:
import torch, os, sys, pathlib, json, glob, shutil, random
import numpy as np, matplotlib.pyplot as plt

# GPU
print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}  |  {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# Repo
REPO_DIR = "/content/261-mini2"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/gracee-chen/261-mini2.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
os.chdir(REPO_DIR)

# Dependencies
!pip install -q segmentation-models-pytorch timm transformers scipy
!pip install -q git+https://github.com/facebookresearch/sam2.git

# Find VOC dataset (reuse previous download)
VOC_ROOT = None
for candidate in [
    os.path.join(REPO_DIR, "voc_data"),
    os.path.join(REPO_DIR, "voc_data", "VOCtrainval_06-Nov-2007"),
    os.path.join(REPO_DIR, "VOCtrainval_06-Nov-2007"),
]:
    if os.path.isdir(os.path.join(candidate, "VOCdevkit", "VOC2007", "JPEGImages")):
        VOC_ROOT = candidate; break
if VOC_ROOT is None:
    for p in pathlib.Path(REPO_DIR).rglob("VOCdevkit"):
        if (p / "VOC2007" / "JPEGImages").is_dir():
            VOC_ROOT = str(p.parent); break
if VOC_ROOT is None:
    print("Dataset not found. Downloading from Kaggle...")
    !pip install -q kaggle
    from google.colab import files; files.upload()
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    EXTRACT = os.path.join(REPO_DIR, "voc_data")
    !kaggle datasets download -d zaraks/pascal-voc-2007 -p /tmp/voc && unzip -q /tmp/voc/pascal-voc-2007.zip -d {EXTRACT} && rm -rf /tmp/voc
    for c in [EXTRACT, os.path.join(EXTRACT, "VOCtrainval_06-Nov-2007")]:
        if os.path.isdir(os.path.join(c, "VOCdevkit", "VOC2007", "JPEGImages")):
            VOC_ROOT = c; break

assert VOC_ROOT, "Could not find VOCdevkit/VOC2007/JPEGImages"
print(f"VOC_ROOT: {VOC_ROOT}  |  Images: {len(os.listdir(os.path.join(VOC_ROOT,'VOCdevkit','VOC2007','JPEGImages')))}")

# Paths
CKPT = "checkpoints_v4"
RES  = "results_v4"
IMG_SIZE = 512
SAM2_CKPT = os.path.join(REPO_DIR, "sam2.1_hiera_base_plus.pt")
SAM2_CFG  = "configs/sam2.1/sam2.1_hiera_b+.yaml"
if not os.path.exists(SAM2_CKPT):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_base_plus.pt -O {SAM2_CKPT}
os.makedirs(RES, exist_ok=True)
print(f"Checkpoints: {CKPT}/  |  Results: {RES}/  |  Image size: {IMG_SIZE}")

## 1. Dataset Exploration

In [ ]:
sys.path.insert(0, REPO_DIR)
from dataset.voc_dataset import VOC_CLASSES, NUM_CLASSES, get_datasets, mask_to_class_index
from torch.utils.data import DataLoader

train_ds, val_ds = get_datasets(VOC_ROOT, image_size=IMG_SIZE)
print(f"Train: {len(train_ds)}  |  Val: {len(val_ds)} (test set)  |  Classes: {NUM_CLASSES}")

explore_dir = f"{RES}/exploration"; os.makedirs(explore_dir, exist_ok=True)
mean = torch.tensor([.485,.456,.406]).view(3,1,1)
std  = torch.tensor([.229,.224,.225]).view(3,1,1)

# --- 1. Sample visualisation ---
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for col, idx in enumerate(random.sample(range(len(train_ds)), 4)):
    img, mask = train_ds[idx]
    img_np = (img*std+mean).clamp(0,1).permute(1,2,0).numpy()
    mask_np = mask_to_class_index(mask).numpy(); md = mask_np.copy(); md[md==255]=0
    axes[0,col].imshow(img_np); axes[0,col].set_title(f"#{idx}"); axes[0,col].axis("off")
    axes[1,col].imshow(md, cmap="tab20", vmin=0, vmax=20); axes[1,col].axis("off")
    axes[1,col].set_title(", ".join([VOC_CLASSES[c] for c in np.unique(mask_np) if c<21]))
plt.suptitle("Training Samples: Image + Segmentation Mask", fontsize=14); plt.tight_layout()
plt.savefig(f"{explore_dir}/samples.png", dpi=150, bbox_inches="tight"); plt.show()

# --- 2. Pixel-level + Image-level class distribution (side by side) ---
counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for _, masks in DataLoader(train_ds, batch_size=4):
    for m in masks: v = mask_to_class_index(m).numpy(); counts += np.bincount(v[v!=255], minlength=NUM_CLASSES)
pcts = 100*counts/counts.sum()

img_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for i in range(len(train_ds)):
    _, mask = train_ds[i]
    for c in np.unique(mask_to_class_index(mask).numpy()):
        if c < NUM_CLASSES: img_counts[c] += 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 6))
colors = plt.cm.tab20(np.linspace(0,1,21))

bars1 = ax1.bar(range(NUM_CLASSES), pcts, color=colors)
ax1.set_xticks(range(NUM_CLASSES)); ax1.set_xticklabels(VOC_CLASSES, rotation=45, ha="right", fontsize=8)
ax1.set_ylabel("Pixel %"); ax1.set_title("Pixel-level Class Distribution"); ax1.grid(axis="y", alpha=.3)
for b, p in zip(bars1, pcts):
    if p > 0.3: ax1.text(b.get_x()+b.get_width()/2, b.get_height(), f"{p:.1f}%", ha="center", va="bottom", fontsize=6)

bars2 = ax2.bar(range(NUM_CLASSES), img_counts, color=colors)
ax2.set_xticks(range(NUM_CLASSES)); ax2.set_xticklabels(VOC_CLASSES, rotation=45, ha="right", fontsize=8)
ax2.set_ylabel("# Images"); ax2.set_title(f"Image-level Class Frequency ({len(train_ds)} images)"); ax2.grid(axis="y", alpha=.3)
for b, c in zip(bars2, img_counts):
    if c > 0: ax2.text(b.get_x()+b.get_width()/2, b.get_height(), str(c), ha="center", va="bottom", fontsize=7)

plt.suptitle("Class Distribution (train)", fontsize=13); plt.tight_layout()
plt.savefig(f"{explore_dir}/class_distribution.png", dpi=150, bbox_inches="tight"); plt.show()

# --- 3. Image size distribution (original sizes before resize) ---
from torchvision.datasets import VOCSegmentation
raw_ds = VOCSegmentation(root=VOC_ROOT, year="2007", image_set="train", download=False)
widths, heights = [], []
for i in range(len(raw_ds)):
    img, _ = raw_ds[i]
    widths.append(img.size[0]); heights.append(img.size[1])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=30, color="steelblue", edgecolor="white"); axes[0].set_xlabel("Width"); axes[0].set_ylabel("Count"); axes[0].set_title("Original Image Width Distribution")
axes[1].hist(heights, bins=30, color="coral", edgecolor="white"); axes[1].set_xlabel("Height"); axes[1].set_title("Original Image Height Distribution")
plt.suptitle(f"Image Size Distribution (n={len(raw_ds)})", fontsize=12); plt.tight_layout()
plt.savefig(f"{explore_dir}/image_size_dist.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"Width:  min={min(widths)}, max={max(widths)}, mean={np.mean(widths):.0f}")
print(f"Height: min={min(heights)}, max={max(heights)}, mean={np.mean(heights):.0f}")

# --- 4. Classes per image distribution ---
cls_per_img = []
for i in range(len(train_ds)):
    _, mask = train_ds[i]
    classes = [c for c in np.unique(mask_to_class_index(mask).numpy()) if c < NUM_CLASSES and c > 0]
    cls_per_img.append(len(classes))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(cls_per_img, bins=range(0, max(cls_per_img)+2), color="mediumpurple", edgecolor="white", align="left")
ax.set_xlabel("# Object Classes per Image"); ax.set_ylabel("Count")
ax.set_title(f"Object Classes per Image (mean={np.mean(cls_per_img):.1f})")
ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig(f"{explore_dir}/classes_per_image.png", dpi=150, bbox_inches="tight"); plt.show()

# --- Summary ---
print(f"\n{'='*50}")
print(f"  Dataset Summary")
print(f"{'='*50}")
print(f"  Train images     : {len(train_ds)}")
print(f"  Val images       : {len(val_ds)}")
print(f"  Classes          : {NUM_CLASSES} (20 objects + background)")
print(f"  Ignore label     : 255 (boundary)")
print(f"  Avg classes/image: {np.mean(cls_per_img):.1f}")
print(f"  Avg image size   : {np.mean(widths):.0f} x {np.mean(heights):.0f}")
print(f"  Most common class: {VOC_CLASSES[np.argmax(counts)]} ({pcts[np.argmax(counts)]:.1f}% pixels)")
print(f"  Rarest class     : {VOC_CLASSES[np.argmin(counts[1:])+1]} ({pcts[np.argmin(counts[1:])+1]:.2f}% pixels)")
print(f"{'='*50}")

## 2. Train Models

**v4 changes:** image size 512, CE+Dice (0.7+0.3), DeepLabV3+ OS=8, improved SAM2 head

In [ ]:
# U-Net (ResNet-50) — 512x512, augmentation, CE+Dice
!python train/train_unet.py --voc-root {VOC_ROOT} \
    --epochs 80 --batch-size 8 --lr 1e-4 --image-size {IMG_SIZE} \
    --augment --ce-weight 0.7 --dice-weight 0.3 \
    --checkpoint-dir {CKPT}/unet

# DeepLabV3+ (ResNet-50, OS=8) — 512x512, augmentation, CE+Dice
!python train/train_deeplabv3plus.py --voc-root {VOC_ROOT} \
    --epochs 80 --batch-size 8 --lr 1e-4 --image-size {IMG_SIZE} \
    --augment --ce-weight 0.7 --dice-weight 0.3 \
    --checkpoint-dir {CKPT}/deeplabv3plus

In [ ]:
# DINOv2 (ViT-B/14) — frozen backbone, 224x224 (forced by patch size)
!python train/train_dinov2.py --voc-root {VOC_ROOT} \
    --epochs 80 --batch-size 16 --lr 1e-3 \
    --augment --ce-weight 0.7 --dice-weight 0.3 \
    --checkpoint-dir {CKPT}/dinov2

In [ ]:
# SAM2 (Hiera B+) — frozen encoder, 512x512, CE+Dice
!python train/train_sam2.py --voc-root {VOC_ROOT} \
    --sam2-ckpt {SAM2_CKPT} \
    --sam2-cfg  {SAM2_CFG} \
    --epochs 80 --batch-size 4 --lr 3e-4 --image-size {IMG_SIZE} \
    --augment --ce-weight 0.7 --dice-weight 0.3 \
    --checkpoint-dir {CKPT}/sam2

## 3. Evaluation

In [ ]:
# Detect available models
available = [(n, f"{CKPT}/{n}/best.pth") for n in ["unet","deeplabv3plus","dinov2","sam2"] if os.path.exists(f"{CKPT}/{n}/best.pth")]
models = [a[0] for a in available]; ckpts = [a[1] for a in available]
print(f"Available: {models}")

mt = " ".join(models); cp = " ".join(ckpts)
# IMPORTANT: pass both --sam2-ckpt AND --sam2-cfg to avoid config mismatch
sf = f"--sam2-ckpt {SAM2_CKPT} --sam2-cfg {SAM2_CFG}" if "sam2" in models else ""

# Compute metrics (image-size 512 for most models; DINOv2 auto-uses 224)
!python evaluation/metrics/compute_metrics.py \
    --model-type {mt} --checkpoint {cp} \
    --voc-root {VOC_ROOT} --image-size {IMG_SIZE} \
    --output-dir {RES}/metrics {sf}

# Cross-model comparison
cd = " ".join([f"{CKPT}/{m}" for m in models])
!python evaluation/compare.py --models {mt} --checkpoint-dirs {cd} --metrics-dir {RES}/metrics --output-dir {RES}/compare

# Print summary table
t = f"{RES}/compare/summary_table.txt"
if os.path.exists(t): print(open(t).read())

In [ ]:
# Per-model metrics tables
for name in models:
    fpath = f"{RES}/metrics/{name}_metrics.json"
    if not os.path.exists(fpath): continue
    m = json.load(open(fpath))
    hd = f"{m['HD95']:.2f}" if not np.isnan(m['HD95']) else 'N/A'
    print(f"\n{'='*55}")
    print(f"  {name.upper()}  |  mIoU={m['mIoU']:.4f}  mDice={m['mDice']:.4f}  PixAcc={m['pixel_accuracy']:.4f}  HD95={hd}")
    print(f"{'='*55}")
    print(f"  {'Class':<15} {'IoU':>8} {'Acc':>8} {'Dice':>8}")
    print(f"  {'-'*42}")
    for i, cls in enumerate(VOC_CLASSES):
        def fmt(v): return f"{v:.4f}" if not np.isnan(v) else "  --  "
        print(f"  {cls:<15} {fmt(m['iou_per_class'][i]):>8} {fmt(m['acc_per_class'][i]):>8} {fmt(m['dice_per_class'][i]):>8}")

In [ ]:
# Confusion matrix + Inference + Mosaic + Top3/Worst3
sf_sam2 = f"--sam2-ckpt {SAM2_CKPT} --sam2-cfg {SAM2_CFG}"

for name in models:
    sf2 = sf_sam2 if name == "sam2" else ""
    !python evaluation/metrics/confusion_matrix.py \
        --model-type {name} --checkpoint {CKPT}/{name}/best.pth \
        --voc-root {VOC_ROOT} --image-size {IMG_SIZE} --output-dir {RES}/metrics {sf2}

for name in models:
    sf2 = sf_sam2 if name == "sam2" else ""
    !python inference/infer_{name}.py \
        --checkpoint {CKPT}/{name}/best.pth \
        --voc-root {VOC_ROOT} --num-samples 4 --output-dir {RES}/infer_{name} {sf2}

specs = " ".join([f"{n}:{CKPT}/{n}/best.pth" for n in models])
!python evaluation/visualization/visualize_mosaic.py \
    --voc-root {VOC_ROOT} --models {specs} --num-images 6 \
    --output {RES}/visualization/mosaic.png {sf}

for metric in ["miou", "person"]:
    !python evaluation/visualization/visualize_comparison.py \
        --voc-root {VOC_ROOT} --model-type {mt} --checkpoint {cp} \
        --metric {metric} --output-dir {RES}/visualization {sf}

## 4. Ablation Studies

In [ ]:
!python evaluation/ablation/ablation_backbone.py --voc-root {VOC_ROOT}
!python evaluation/ablation/ablation_loss.py --voc-root {VOC_ROOT}
!python evaluation/ablation/ablation_augmentation.py --voc-root {VOC_ROOT}
!python evaluation/ablation/ablation_pretrain.py --voc-root {VOC_ROOT}
!python evaluation/ablation/ablation_resolution.py --voc-root {VOC_ROOT}

## 5. Display All Plots & Download

In [ ]:
from IPython.display import display, Image as IPImage

all_plots = []
for d in [f"{RES}/exploration", f"{RES}/compare", f"{RES}/metrics", f"{RES}/visualization",
          f"{RES}/infer_unet", f"{RES}/infer_deeplabv3plus", f"{RES}/infer_dinov2", f"{RES}/infer_sam2",
          "results/ablation"]:
    all_plots.extend(sorted(glob.glob(os.path.join(d, "**", "*.png"), recursive=True)))

print(f"Total plots: {len(all_plots)}\n")
for f in all_plots:
    print(f"--- {f} ---"); display(IPImage(filename=f, width=800)); print()

In [ ]:
from google.colab import files
zip_path = "/content/261-mini2-results-v4"
shutil.make_archive(zip_path, "zip", root_dir=REPO_DIR, base_dir=RES)
print(f"Created: {zip_path}.zip")
files.download(f"{zip_path}.zip")